# 03 - Backtest: Elo-Poisson vs XGBoost Expected Goals

Stage 9 compares the existing pure Elo-Poisson pipeline with the Phase 2 XGBoost expected-goals model. Both variants feed expected goals into the same `score_matrix -> outcome_probs -> best_score` stack and use identical scoring metrics.

## No-Leakage Protocol

Elo-Poisson recomputes ratings strictly before each held-out event. XGBoost uses rows from `data/processed/feature_matrix.parquet`, which were built in one chronological pass, and the saved model metadata confirms training used matches before `2022-01-01`. Every held-out row must be on or after that split.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from src.backtest import run_xgb_comparison_backtest

In [2]:
predictions, summary, leakage = run_xgb_comparison_backtest()

print("Leakage checks:")
print(leakage.to_string(index=False))
assert leakage["no_elo_leakage"].all()
assert leakage["xgb_all_test_rows_after_split"].all()

Leakage checks:
                      event rating_cutoff max_rating_train_date xgb_train_split  xgb_all_test_rows_after_split  no_elo_leakage  test_matches
        2022 FIFA World Cup    2022-11-20            2022-11-19      2022-01-01                           True            True            64
             UEFA Euro 2024    2024-06-14            2024-06-12      2022-01-01                           True            True            51
          Copa América 2024    2024-06-20            2024-06-19      2022-01-01                           True            True            32
African Cup of Nations 2023    2024-01-13            2024-01-12      2022-01-01                           True            True            52


## Side-By-Side Metrics

Rows include each held-out event and a pooled block. Higher is better for W/D/L accuracy, exact-score rate, and total points; lower is better for Brier and log-loss.

In [3]:
display_cols = [
    "event",
    "model",
    "matches",
    "wdl_accuracy",
    "exact_score_rate",
    "total_points",
    "brier",
    "log_loss",
]
comparison = summary[display_cols].copy()
for col in ["wdl_accuracy", "exact_score_rate", "brier", "log_loss"]:
    comparison[col] = comparison[col].round(3)
comparison

,event,model,matches,wdl_accuracy,exact_score_rate,total_points,brier,log_loss
0,2022 FIFA World Cup,Elo-Poisson,64,0.516,0.047,1605,0.603,1.015
1,2022 FIFA World Cup,XGBoost,64,0.516,0.062,1640,0.627,1.049
2,African Cup of Nations 2023,Elo-Poisson,52,0.442,0.115,1220,0.692,1.132
3,African Cup of Nations 2023,XGBoost,52,0.462,0.192,1360,0.666,1.095
4,Copa América 2024,Elo-Poisson,32,0.594,0.125,920,0.532,0.901
5,Copa América 2024,XGBoost,32,0.594,0.125,930,0.534,0.908
6,Pooled,Elo-Poisson,199,0.503,0.131,5170,0.622,1.036
7,Pooled,XGBoost,199,0.518,0.136,5345,0.620,1.032
8,UEFA Euro 2024,Elo-Poisson,51,0.490,0.255,1425,0.630,1.049
9,UEFA Euro 2024,XGBoost,51,0.529,0.176,1415,0.617,1.024


## Verdict

The table below computes pooled deltas as `XGBoost - Elo-Poisson`. This is the honest comparison: XGBoost should only be considered better where it improves out-of-sample metrics, not because it is a more complex model.

In [4]:
metric_cols = ["wdl_accuracy", "exact_score_rate", "total_points", "brier", "log_loss"]
pooled = summary.loc[summary["event"] == "Pooled"].set_index("model")
delta = pooled.loc["XGBoost", metric_cols] - pooled.loc["Elo-Poisson", metric_cols]
print("Pooled XGBoost - Elo-Poisson deltas:")
print(delta.to_string())

print("\nHonest verdict:")
print(
    "XGBoost improves pooled W/D/L accuracy, exact-score rate, total points, Brier, "
    "and log-loss in this out-of-sample 2022+ comparison. The gains are real but "
    "modest: points improve by 175 over 199 matches, W/D/L accuracy by about 1.5 "
    "percentage points, and Brier by about 0.0017. By event, XGBoost helps most on "
    "AFCON and Euro calibration, is roughly tied on Copa, and has worse Brier/log-loss "
    "than Elo-Poisson at the 2022 World Cup despite slightly more points."
)

Pooled XGBoost - Elo-Poisson deltas:
wdl_accuracy        0.015075
exact_score_rate    0.005025
total_points             175
brier              -0.001684
log_loss           -0.003846

Honest verdict:
XGBoost improves pooled W/D/L accuracy, exact-score rate, total points, Brier, and log-loss in this out-of-sample 2022+ comparison. The gains are real but modest: points improve by 175 over 199 matches, W/D/L accuracy by about 1.5 percentage points, and Brier by about 0.0017. By event, XGBoost helps most on AFCON and Euro calibration, is roughly tied on Copa, and has worse Brier/log-loss than Elo-Poisson at the 2022 World Cup despite slightly more points.
